# **Task 02 — Data Quality Checking & Preprocessing**

## Objective

Inspect the concrete dataset for missing values,
duplicate records, invalid measurements, and
potential outliers.

Prepare a consistent dataset for further analysis
without unnecessarily modifying the original data.


In [ ]:
import pandas as pd
from pathlib import Path
# Support running from project root or notebooks folder
paths = [    
    Path("data/raw/Concrete_Data.xls"),    
    Path("../data/raw/Concrete_Data.xls")
]
data_path = next((p for p in paths if p.exists()), None)
if data_path is None:    
    raise FileNotFoundError("Concrete_Data.xls not found.")
# Load original data
df = pd.read_excel(data_path,engine="xlrd")
print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (1030, 9)


C:\Users\WW\AppData\Local\Temp\ipykernel_4680\836804605.py:12: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  df = pd.read_excel(data_path,engine="xlrd")


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [2]:
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)
print("\nTotal missing values:", missing_values.sum())


Missing values per column:
Cement (component 1)(kg in a m^3 mixture)                0
Blast Furnace Slag (component 2)(kg in a m^3 mixture)    0
Fly Ash (component 3)(kg in a m^3 mixture)               0
Water  (component 4)(kg in a m^3 mixture)                0
Superplasticizer (component 5)(kg in a m^3 mixture)      0
Coarse Aggregate  (component 6)(kg in a m^3 mixture)     0
Fine Aggregate (component 7)(kg in a m^3 mixture)        0
Age (day)                                                0
Concrete compressive strength(MPa, megapascals)          0
dtype: int64

Total missing values: 0


In [5]:
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)
# Display repeated records
display(df[df.duplicated(keep=False)].head(10))


Exact duplicate rows: 25


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
72,425.0,106.3,0.0,153.5,16.5,852.1,887.1,3,33.398217
77,425.0,106.3,0.0,153.5,16.5,852.1,887.1,3,33.398217
80,425.0,106.3,0.0,153.5,16.5,852.1,887.1,3,33.398217
83,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
86,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
88,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
91,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
95,425.0,106.3,0.0,153.5,16.5,852.1,887.1,7,49.201007
100,425.0,106.3,0.0,153.5,16.5,852.1,887.1,7,49.201007
103,425.0,106.3,0.0,153.5,16.5,852.1,887.1,7,49.201007


In [6]:
# Count negative values in each column
negative_counts = (df < 0).sum()
print("Negative values:")
print(negative_counts)
# Check invalid age and strength
invalid_age = (df.iloc[:, 7] <= 0).sum()
invalid_strength = (df.iloc[:, 8] <= 0).sum()
print("\nInvalid ages:", invalid_age)
print("Invalid strengths:", invalid_strength)


Negative values:
Cement (component 1)(kg in a m^3 mixture)                0
Blast Furnace Slag (component 2)(kg in a m^3 mixture)    0
Fly Ash (component 3)(kg in a m^3 mixture)               0
Water  (component 4)(kg in a m^3 mixture)                0
Superplasticizer (component 5)(kg in a m^3 mixture)      0
Coarse Aggregate  (component 6)(kg in a m^3 mixture)     0
Fine Aggregate (component 7)(kg in a m^3 mixture)        0
Age (day)                                                0
Concrete compressive strength(MPa, megapascals)          0
dtype: int64

Invalid ages: 0
Invalid strengths: 0


In [8]:
# Check Cement, Water, Age, and Strength
columns_to_check = [    df.columns[0],    df.columns[3],    df.columns[7],    df.columns[8]]
selected = df[columns_to_check]
Q1 = selected.quantile(0.25)
Q3 = selected.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = (    (selected < lower_bound) |   (selected > upper_bound))
print("Potential outliers percolumn:")
print(outliers.sum())



Potential outliers percolumn:
Cement (component 1)(kg in a m^3 mixture)            0
Water  (component 4)(kg in a m^3 mixture)            9
Age (day)                                           59
Concrete compressive strength(MPa, megapascals)      4
dtype: int64


In [9]:
print(df.columns.tolist())


['Cement (component 1)(kg in a m^3 mixture)', 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)', 'Fly Ash (component 3)(kg in a m^3 mixture)', 'Water  (component 4)(kg in a m^3 mixture)', 'Superplasticizer (component 5)(kg in a m^3 mixture)', 'Coarse Aggregate  (component 6)(kg in a m^3 mixture)', 'Fine Aggregate (component 7)(kg in a m^3 mixture)', 'Age (day)', 'Concrete compressive strength(MPa, megapascals) ']


In [11]:
for col in columns_to_check:
    flagged = df.loc[outliers[col], col]
    print(f"\n{col}")
    print("Outlier count:", len(flagged))
    print("Minimum flagged value:", flagged.min() if len(flagged) else "None")
    print("Maximum flagged value:", flagged.max() if len(flagged) else "None")




Cement (component 1)(kg in a m^3 mixture)
Outlier count: 0
Minimum flagged value: None
Maximum flagged value: None

Water  (component 4)(kg in a m^3 mixture)
Outlier count: 9
Minimum flagged value: 121.75
Maximum flagged value: 247.0

Age (day)
Outlier count: 59
Minimum flagged value: 180
Maximum flagged value: 365

Concrete compressive strength(MPa, megapascals) 
Outlier count: 4
Minimum flagged value: 79.98611076
Maximum flagged value: 82.5992248


In [16]:
# Keep original data unchanged
df_clean = df.copy()
# Verify expected number of columns
assert df_clean.shape[1] == 9
# Rename columns using the verified UCI order
df_clean.columns = [    "cement",    "blast_furnace_slag",    "fly_ash",    "water",    "superplasticizer",    "coarse_aggregate",    "fine_aggregate",    "age",    "strength"]
# Ensure all columns contain numeric values
for column in df_clean.columns:    
    df_clean[column] = pd.to_numeric(        df_clean[column], errors="raise"    )
display(df_clean.head())


,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [19]:
assert df_clean.isnull().sum().sum() == 0
assert (df_clean < 0).sum().sum() == 0
assert (df_clean["age"] > 0).all()
assert (df_clean["strength"] > 0).all()
print("Basic data quality checks passed!")


Basic data quality checks passed!


In [20]:
# Build output path from original dataset location
output_path = (    data_path.parent.parent /    "processed" /    "concrete_clean.csv")
# Ensure processed folder exists
output_path.parent.mkdir(    
parents=True,    
exist_ok=True
)
# Save processed data
df_clean.to_csv(output_path, index=False)
print("Dataset saved to:", output_path)


Dataset saved to: ..\data\processed\concrete_clean.csv


In [21]:
saved_df = pd.read_csv(output_path)
print("Saved dataset shape:", saved_df.shape)
display(saved_df.head())


Saved dataset shape: (1030, 9)


,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


## Data Quality Findings

### 1. Missing Values
- Total missing values: 0
- Action taken: The UCI documentation reports no missing values, so we expect zero in every column. However, we verify this ourselves.

### 2. Duplicate Records
- Exact duplicate records: 25
- Action taken: Retained for further investigation.

### 3. Invalid Values
- Negative values: 0
- Invalid age values: 0
- Invalid strength values: 0

### 4. Potential Outliers
- Outliers were identified using the IQR method.
- No outliers were automatically removed.

### 5. Preprocessing
- Column names were standardized.
- Numerical data types were verified.
- Original dataset was preserved.

### Conclusion
The dataset was inspected for quality issues
and prepared for statistical analysis.
Any unresolved issues should be documented.
